In [22]:
import torch

In [23]:
sample_rate = 22050
frequency = 440
duration = 1

In [24]:
waveform_length = sample_rate * duration
sample_indices = torch.arange(waveform_length, dtype=torch.float32)

In [25]:
time = sample_indices / sample_rate

In [26]:
cycles = frequency * time

In [27]:
amplitude = torch.sin(2 * torch.pi * cycles)

In [28]:
waveform = amplitude.unsqueeze(0)

current_samples = amplitude[:-1]
next_samples = amplitude[1:]
upward_crossing_count = ((current_samples < 0) & (next_samples >= 0)).sum().item()

print("Waveform shape:", waveform.shape)
print("Waveform dtype:", waveform.dtype)
print("Minimum amplitude:", waveform.min().item())
print("Maximum amplitude:", waveform.max().item())
print("Upward zero crossings:", upward_crossing_count)

Waveform shape: torch.Size([1, 22050])
Waveform dtype: torch.float32
Minimum amplitude: -0.9999997615814209
Maximum amplitude: 0.9999998211860657
Upward zero crossings: 439


In [29]:
assert waveform.shape == (1, waveform_length)
assert waveform.dtype == torch.float32
assert torch.all(waveform >= -1.0)
assert torch.all(waveform <= 1.0)
assert abs(upward_crossing_count - frequency * duration) <= 1

print("Task 1 passed: controlled 440 Hz waveform verified.")

Task 1 passed: controlled 440 Hz waveform verified.


In [30]:
n_fft       = 1024
win_length  = 1024
hop_length  = 256
center      = True

In [ ]:
window_duration_ms = win_length / sample_rate * 1000
hop_duration_ms = hop_length / sample_rate * 1000
overlap_samples = win_length - hop_length
overlap_percentage = overlap_samples / win_length * 100

print("Window duration:", window_duration_ms, "ms")
print("Hop duration:", hop_duration_ms, "ms")
print("Overlap:", overlap_samples, "samples")
print("Overlap:", overlap_percentage, "%")

In [ ]:
frequency_bins = n_fft // 2 + 1
bin_spacing_hz = sample_rate / n_fft

print("Frequency-bin count:", frequency_bins)
print("Bin spacing:", bin_spacing_hz, "Hz")

In [ ]:
bin_index = 20
bin_20_hz = bin_index * bin_spacing_hz
error_from_440_hz = abs(frequency - bin_20_hz)

print("Bin 20 center:", bin_20_hz, "Hz")
print("Distance from 440 Hz:", error_from_440_hz, "Hz")

In [ ]:
assert frequency_bins == 513
assert abs(bin_spacing_hz - 21.533203125) < 1e-9
assert error_from_440_hz < bin_spacing_hz

print("Frequency-grid prediction passed.")

In [ ]:
padding = n_fft // 2 if center else 0

In [37]:
effective_length = waveform_length + 2 * padding
predicted_frames = 1 + (effective_length - n_fft) // hop_length
predicted_frequency_bins = n_fft // 2 + 1

predicted_stft_shape = (
    predicted_frequency_bins,
    predicted_frames,
)


In [38]:
print("padding:", padding)
print("effective_length:", effective_length)
print("predicted_frames:", predicted_frames)
print("predicted_stft_shape:", predicted_stft_shape)

padding: 512
effective_length: 23074
predicted_frames: 87
predicted_stft_shape: (513, 87)


In [ ]:
batch_size = waveform.shape[0]